In [1]:
# This notebook is a work in progress
_cf_value_cache = {}

def cf_value(terms):
    # return continued_fraction of terms as a real number (cached)
    key = tuple(tuple(part) for part in terms)
    if key not in _cf_value_cache:
        _cf_value_cache[key] = continued_fraction(terms).n()
    return _cf_value_cache[key]

def cf_extrema(term_lists):
    # return (min_value, min_terms), (max_value, max_terms) for a list of CF term lists
    valued = [(cf_value(terms), terms) for terms in term_lists]
    return (
        min(valued, key=lambda item: item[0]),
        max(valued, key=lambda item: item[0]),
    )

def sr_extrema(sr_candidates, component):
    # component: 0 for s, 1 for r; sr_candidates is e.g. sr_1[j] or sr_2[j]
    return cf_extrema(pair[component] for pair in sr_candidates)

def check_ii(h, k, sr_begin, sr_end):
    # verify connectivity condition (ii) between sr_1[0] and sr_2[-1]
    (s_lo, _), (s_hi, _) = sr_extrema(sr_begin, 0)
    (r_lo, _), (r_hi, _) = sr_extrema(sr_begin, 1)
    (s2_lo, _), (s2_hi, _) = sr_extrema(sr_end, 0)
    (r2_lo, _), (r2_hi, _) = sr_extrema(sr_end, 1)

    s_ok = (s_hi <= s2_lo) if h % 2 == 0 else (s_lo >= s2_hi)
    r_ok = (r_hi <= r2_lo) if k % 2 == 0 else (r_lo >= r2_hi)
    return {"s": s_ok, "r": r_ok}

In [ ]:
# how to do it?

# 2.1: 1 = (-1)^{h+k} 1 < v(A) < 1.98 (i)
successors = [
    ([2], [2]),
    ([3], [1, 1]),
    ([2,3], [2,3]),
    ([1, 1, 3], [3, 1]),
    ([1], []),
]

# h:even k:even
h = 0
k = 0

etas = [(3,1), (1,3)]
thetas = [(2,1), (1,2)]

# initial point of the T-interval : [0;a_1, ... , a_k + r_1^j] + [0;a_{-1}, ... , a_{-h} + s_1^j] 
# end point of the T-interval : [0;a_1, ... , a_k + r_2^j] + [0;a_{-1}, ... , a_{-h} + s_2^j] 

sr_1 = [] # list of candidates of (s_1^j, r_1^j)
sr_2 = [] # list of candidates of (s_2^j, r_2^j)

for successor in successors:
    sr_1.append((
                ([(0, *successor[0]), etas[len(successor[0]) % 2]], [(0, *successor[1]), thetas[len(successor[1]) % 2]]),
                ([(0, *successor[0]), thetas[len(successor[0]) % 2]], [(0, *successor[1]), etas[len(successor[1]) % 2]])
                ))
    sr_2.append((
                ([(0, *successor[0]), etas[(len(successor[0]) + 1) % 2]], [(0, *successor[1]), thetas[(len(successor[1]) + 1) % 2]]),
                ([(0, *successor[0]), thetas[(len(successor[0]) + 1) % 2]], [(0, *successor[1]), etas[(len(successor[1]) + 1) % 2]])
                ))
# extrema of s/r candidates as continued fractions
# sr_1[0]: initial endpoint (j=1), sr_2[-1]: final endpoint (j=n)
for label, sr in [("sr_1[0]", sr_1[0]), ("sr_2[-1]", sr_2[-1])]:
    for name, component in [("s", 0), ("r", 1)]:
        (lo_val, lo_terms), (hi_val, hi_terms) = sr_extrema(sr, component)
        print(f"{label} {name}: min = {lo_val}, max = {hi_val}")
        print(f"  min: {lo_terms}")
        print(f"  max: {hi_terms}")

# check (i)
# this condition seems trivial. in this case, \tilde{s}_1^j ,\tilde{r}_1^j are something like [0;(3,1)] and  [0;(2,1)]

# check (ii)
ii = check_ii(h, k, sr_1[0], sr_2[-1])
print(f"check (ii) s: {ii['s']}, r: {ii['r']}, all: {all(ii.values())}")

# c_-, c_+ \in [0.2, 0.9]のときのH(s_1^j, s_2^{j-1}, r_2^{j-1}, r_1^j, c_-, c_+)の範囲を求める←これはすぐできそう？

# (iii)の条件を満たすv(A)の範囲を求める


sr_1[0] s: min = 0.358257569495584, max = 0.366025403784439
  min: [(0, 2), (1, 3)]
  max: [(0, 2), (1, 2)]
sr_1[0] r: min = 0.358257569495584, max = 0.366025403784439
  min: [(0, 2), (1, 3)]
  max: [(0, 2), (1, 2)]
sr_2[-1] s: min = 0.732050807568877, max = 0.791287847477920
  min: [(0, 1), (2, 1)]
  max: [(0, 1), (3, 1)]
sr_2[-1] r: min = 0.732050807568877, max = 0.791287847477920
  min: [(0,), (1, 2)]
  max: [(0,), (1, 3)]


In [12]:
print(RR(continued_fraction([(0,1,4),(1,3)])) - RR(continued_fraction([(0,1,4),(3,1)])))
print(RR(continued_fraction([(0,3,1),(1,3)])) - RR(continued_fraction([(0,3,1),(3,1)])))

print(RR(continued_fraction([(0,1,4),(1,3)])) - RR(continued_fraction([(0,1,4),(1,2)])))
print(RR(continued_fraction([(0,3,1),(1,3)])) - RR(continued_fraction([(0,3,1),(1,2)])))

print(RR(continued_fraction([(0,1,4),(2,1)])) - RR(continued_fraction([(0,1,4),(3,1)])))
print(RR(continued_fraction([(0,3,1),(2,1)])) - RR(continued_fraction([(0,3,1),(3,1)])))

0.0173050074306340
0.0172738129510922
0.00178446554099798
0.00149992137586852
0.00362050047584750
0.00418657660514937
